In [ ]:
from experta import*

class Person(Fact):
    #الشخص الذي يعبر الجسر
    pass

class State(Fact):
    #حالة الجسر
    pass

class Move(Fact):
    #خطوة بين حالتين
    pass

class BridgeExpertSystem(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        #مجموعة لحفظ الحالات التي زرتها
        self.visited_states = set() 
    @DefFacts()
    def initial_state(self):
        yield State(
            left=('me', 'lab', 'worker', 'scientist'),
            right=(),
            light='left',
            time=0,
            path=[]
        )

    #قاعدة عبور شخصين من اليسار لليمين
    #هذه القاعدة تعمل فقط عندما يكون المصباح على الجهة اليسرى
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path),salience=0)
    def move_left_to_right(self, left, right, time, path):
        persons_times = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}

        left_list = list(left)
        n = len(left_list)

        for i in range(n):
            for j in range(i + 1, n):
                p1 = left_list[i]
                p2 = left_list[j]

                new_left = left_list.copy()
                new_left.remove(p1)
                new_left.remove(p2)

                new_right = list(right) + [p1, p2]

                t = max(persons_times[p1], persons_times[p2])
                total_time = time + t

                if total_time <= 17:
                    #لازم المصباح يضل معي انا بس
                    if 'me' in new_right:
                        new_light = 'right'
                    else:
                        new_light = 'left'
                    state_signature = (tuple(sorted(new_left)), tuple(sorted(new_right)), new_light)

                    if state_signature in self.visited_states:
                        continue  # الحالة مكررة، نتجاهلها
                    else:
                        self.visited_states.add(state_signature)
                        print(f" Generated state: Left={new_left}, Right={new_right}, Light={new_light}, Time={total_time}")
                        old_path = list(path)              # 1. حوِّل الـ tuple إلى list
                        step = f"{p1} and {p2} crossed to right in {t} min"
                        new_path = old_path + [step] 
                        self.declare(State(
                            left=tuple(new_left),
                            right=tuple(new_right),
                            light=new_light,
                            time=total_time,
                            path=tuple(new_path) 
                        ))



    #قاعدة العودة من اليمين الى اليسار لشخص واحد, عودة المصباح من اليمين الى اليسار
    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path),salience=0)
    def move_right_to_left(self, left, right, time, path):
        persons_times = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}

        right_list = list(right)

        for p in right_list:
            new_right = right_list.copy()
            new_right.remove(p)

            new_left = list(left) + [p]

            t = persons_times[p]
            total_time = time + t

            if total_time <= 17:
                if 'me' in new_left:
                    new_light = 'left'
                else:
                    continue  
                state_signature = (tuple(sorted(new_left)), tuple(sorted(new_right)), new_light)

                if state_signature in self.visited_states:
                    continue  # الحالة مكررة، نتجاهلها
                else:
                    self.visited_states.add(state_signature)
                    print(f" Generated state: Left={new_left}, Right={new_right}, Light={new_light}, Time={total_time}")
                    old_path = list(path)
                    step = f"{p} returned to left in {t} min"
                    new_path = old_path + [step]
                    self.declare(State(
                        left=tuple(new_left),
                        right=tuple(new_right),
                        light=new_light,
                        time=total_time,
                        path=tuple(new_path)

                    ))
        #قاعدة التحقق من الوصول للحالة الهدف وطباعة مسار الحل
        @Rule(
                State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path),
                TEST(lambda left, right: set(left) == set() and set(right) == {'me', 'lab', 'worker', 'scientist'}),
                salience=100
            )
        def goal_reached(self, left, right, time, path):
            print("\n تم الوصول إلى الهدف خلال", time, "دقيقة!\n")
            print(" خطوات الحل:")
            for step in path:
                print(" ", step)
            self.halt()


engine = BridgeExpertSystem()
engine.reset()
engine.run()


 Generated state: Left=['worker', 'scientist'], Right=['me', 'lab'], Light=right, Time=2
 Generated state: Left=['lab', 'scientist'], Right=['me', 'worker'], Light=right, Time=5
 Generated state: Left=['lab', 'worker'], Right=['me', 'scientist'], Light=right, Time=10
 Generated state: Left=['me', 'scientist'], Right=['lab', 'worker'], Light=left, Time=5
 Generated state: Left=['me', 'worker'], Right=['lab', 'scientist'], Light=left, Time=10
 Generated state: Left=['me', 'lab'], Right=['worker', 'scientist'], Light=left, Time=10
 Generated state: Left=[], Right=['worker', 'scientist', 'me', 'lab'], Light=right, Time=12
 Generated state: Left=['me'], Right=['worker', 'scientist', 'lab'], Light=left, Time=13
 Generated state: Left=['lab', 'worker', 'me'], Right=['scientist'], Light=left, Time=11
 Generated state: Left=['worker'], Right=['scientist', 'lab', 'me'], Light=right, Time=13
 Generated state: Left=['lab'], Right=['scientist', 'worker', 'me'], Light=right, Time=16
 Generated state